# Segment 2: Claude Models

## Claude API for Python Developers

In this segment, we'll cover:
- Basic Usage of Claude Models
- Input Formatting Best Practices
- Multi-Step Prompts and Conversations
- Document Summarization Techniques


## Setup and Imports


In [ ]:
import anthropic
from IPython.display import display, Markdown

# Initialize the client
client = anthropic.Anthropic()

# Helper function to display responses nicely
def show_response(response):
    """Display Claude's response with markdown formatting."""
    display(Markdown(response.content[0].text))

print("Client initialized successfully!")


---
## 1. Basic Usage

### Model Comparison

Claude offers different models optimized for various use cases:

| Model | ID | Best For |
|-------|-----|----------|
| Claude 3.5 Sonnet | `claude-sonnet-4-20250514` | Best balance of speed and intelligence |
| Claude 3 Opus | `claude-3-opus-20240229` | Most capable for complex tasks |
| Claude 3.5 Haiku | `claude-3-5-haiku-20241022` | Fast, efficient for high volume |


In [ ]:
# Basic message - simplest possible API call
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "What is machine learning in one sentence?"}
    ]
)

print("Basic Response:")
print(response.content[0].text)


### System Prompts

System prompts set the context, personality, and constraints for Claude's responses.


In [ ]:
# Using system prompts to control behavior
# Example 1: Technical expert
response_expert = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    system="You are a senior Python developer. Explain concepts with code examples when relevant.",
    messages=[
        {"role": "user", "content": "What are decorators?"}
    ]
)

print("🔧 Technical Expert Response:")
show_response(response_expert)


In [ ]:
# Example 2: Friendly teacher
response_teacher = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    system="You are a friendly teacher explaining to a beginner. Use simple language and analogies.",
    messages=[
        {"role": "user", "content": "What are decorators?"}
    ]
)

print("📚 Friendly Teacher Response:")
show_response(response_teacher)


### Assistant Prefilling

You can start Claude's response with specific text to guide the format or direction.


In [ ]:
# Assistant prefilling to force JSON output
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[
        {
            "role": "user", 
            "content": "List 3 programming languages with their main use case."
        },
        {
            "role": "assistant",
            "content": "["  # Start the response with JSON array
        }
    ]
)

# Complete the JSON
json_response = "[" + response.content[0].text
print("Prefilled JSON response:")
print(json_response)


In [ ]:
# Prefilling to set tone/style
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[
        {"role": "user", "content": "Explain recursion."},
        {"role": "assistant", "content": "Recursion is like"}  # Forces analogy-style response
    ]
)

print("Prefilled analogy response:")
print("Recursion is like" + response.content[0].text)


---
## 2. Input Formatting

Proper formatting helps Claude understand your intent and produce better responses.

### XML Tags

Claude was trained to understand XML-style tags for structuring prompts.


In [ ]:
# Using XML tags to structure input
prompt = """
<document>
Python is a high-level programming language known for its clear syntax and readability.
It was created by Guido van Rossum and first released in 1991.
Python supports multiple programming paradigms including procedural, object-oriented, 
and functional programming.
</document>

<task>
Extract the following information from the document:
1. Creator's name
2. Year of first release
3. Programming paradigms supported
</task>

<output_format>
Respond in a structured format with clear labels.
</output_format>
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{"role": "user", "content": prompt}]
)

print("XML-structured response:")
show_response(response)


### Markdown Formatting

Claude handles markdown well for structured inputs.


In [ ]:
# Using markdown formatting
prompt = """
# Task: Code Review

Please review the following Python function:

```python
def calculate_average(numbers):
    total = 0
    for n in numbers:
        total = total + n
    average = total / len(numbers)
    return average
```

## Review Criteria:
- Code correctness
- Error handling
- Pythonic style
- Performance

Please provide specific suggestions for improvement.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{"role": "user", "content": prompt}]
)

print("Code review response:")
show_response(response)


### Few-Shot Examples

Providing examples in your prompt helps Claude understand the desired format.


In [ ]:
# Few-shot prompting for consistent output format
prompt = """
Classify the sentiment of the following texts.

<examples>
Text: "I love this product! Best purchase ever!"
Sentiment: POSITIVE
Confidence: HIGH

Text: "The delivery was okay, nothing special."
Sentiment: NEUTRAL
Confidence: MEDIUM

Text: "Terrible experience, never buying again."
Sentiment: NEGATIVE
Confidence: HIGH
</examples>

Now classify this text:
Text: "The food was delicious but the service was slow."
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=100,
    messages=[{"role": "user", "content": prompt}]
)

print("Few-shot classification:")
print(response.content[0].text)


### Chain of Thought Prompting

Encouraging Claude to "think step by step" improves reasoning quality.


In [ ]:
# Chain of thought prompting
problem = """
A train leaves Station A at 9:00 AM traveling at 60 mph toward Station B.
Another train leaves Station B at 10:00 AM traveling at 80 mph toward Station A.
If the stations are 280 miles apart, at what time will the trains meet?

Think through this step by step before giving your final answer.
"""

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=500,
    messages=[{"role": "user", "content": problem}]
)

print("Chain of thought reasoning:")
show_response(response)


---
## 3. Multi-Step Prompts

### Conversation History

Claude can maintain context across multiple exchanges.


In [ ]:
# Multi-turn conversation
conversation = [
    {"role": "user", "content": "I'm learning Python. What should I start with?"},
    {"role": "assistant", "content": "Great choice! I recommend starting with:\n1. Variables and data types\n2. Control flow (if/else, loops)\n3. Functions\n4. Lists and dictionaries\n\nWould you like me to explain any of these in detail?"},
    {"role": "user", "content": "Yes, explain functions please."},
    {"role": "assistant", "content": "Functions are reusable blocks of code. Here's a simple example:\n\n```python\ndef greet(name):\n    return f'Hello, {name}!'\n\nresult = greet('Alice')  # Returns 'Hello, Alice!'\n```\n\nKey concepts:\n- `def` keyword defines a function\n- Parameters go in parentheses\n- `return` sends back a value\n\nWould you like to see more examples?"},
    {"role": "user", "content": "How do I use default parameters?"}
]

response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=conversation
)

print("Continuing conversation (Claude remembers context):")
show_response(response)


### Building a Simple Chatbot


In [ ]:
class SimpleChatbot:
    """A simple chatbot that maintains conversation history."""
    
    def __init__(self, system_prompt="You are a helpful assistant."):
        self.client = anthropic.Anthropic()
        self.system_prompt = system_prompt
        self.conversation_history = []
    
    def chat(self, user_message):
        """Send a message and get a response."""
        # Add user message to history
        self.conversation_history.append({
            "role": "user",
            "content": user_message
        })
        
        # Get response from Claude
        response = self.client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=300,
            system=self.system_prompt,
            messages=self.conversation_history
        )
        
        # Extract and store assistant response
        assistant_message = response.content[0].text
        self.conversation_history.append({
            "role": "assistant",
            "content": assistant_message
        })
        
        return assistant_message
    
    def reset(self):
        """Clear conversation history."""
        self.conversation_history = []

# Demo the chatbot
bot = SimpleChatbot(system_prompt="You are a Python tutor. Be concise and helpful.")

print("🤖 Chatbot Demo")
print("-" * 40)

exchanges = [
    "What is a list comprehension?",
    "Show me an example",
    "How is that different from a regular loop?"
]

for msg in exchanges:
    print(f"👤 User: {msg}")
    response = bot.chat(msg)
    print(f"🤖 Bot: {response}")
    print("-" * 40)


### Multi-Step Task Processing

Breaking complex tasks into steps and processing them sequentially.


In [ ]:
# Multi-step processing: Analyze, then act
text = """
Our Q3 sales were $2.5M, up 15% from Q2. The marketing team launched 
three new campaigns that drove significant website traffic. Customer 
satisfaction scores improved to 4.5/5. However, we noticed increased 
support ticket volume, suggesting some product issues need attention.
"""

# Step 1: Extract key metrics
step1_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=200,
    messages=[{
        "role": "user",
        "content": f"""
Extract all numerical metrics from this text. Return as a bullet list.

<text>
{text}
</text>
"""
    }]
)

metrics = step1_response.content[0].text
print("Step 1 - Extracted Metrics:")
print(metrics)
print("\n" + "="*50 + "\n")

# Step 2: Generate insights based on extracted metrics
step2_response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Based on these metrics from a quarterly report:

{metrics}

Provide 3 actionable insights for the leadership team.
"""
    }]
)

print("Step 2 - Generated Insights:")
show_response(step2_response)


---
## 4. Document Summarization

Claude excels at summarizing and extracting information from documents.


In [ ]:
# Sample long document (article about climate change)
long_document = """
Climate Change and Its Global Impact: A Comprehensive Overview

Introduction:
Climate change represents one of the most pressing challenges facing humanity in the 21st century. 
The phenomenon, driven primarily by human activities, has far-reaching consequences for ecosystems, 
economies, and societies worldwide. This article examines the causes, effects, and potential 
solutions to this global crisis.

Causes of Climate Change:
The primary driver of modern climate change is the emission of greenhouse gases, particularly 
carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O). These emissions result from burning 
fossil fuels for energy, deforestation, industrial processes, and agricultural practices. Since 
the Industrial Revolution, atmospheric CO2 levels have increased by over 50%, from approximately 
280 parts per million (ppm) to over 420 ppm today.

Environmental Impacts:
The consequences of climate change are already visible across the globe. Average global 
temperatures have risen by approximately 1.1°C since pre-industrial times. This warming has led 
to melting ice caps and glaciers, rising sea levels, more frequent and intense extreme weather 
events, shifts in ecosystems and biodiversity, and altered precipitation patterns affecting 
agriculture.

Economic Implications:
The economic costs of climate change are substantial and growing. The World Bank estimates that 
climate change could push an additional 132 million people into extreme poverty by 2030. 
Industries such as agriculture, insurance, and tourism face significant disruptions. However, 
the transition to clean energy also presents economic opportunities, with renewable energy 
sectors experiencing rapid growth.

Solutions and Mitigation:
Addressing climate change requires a multi-faceted approach. Key strategies include transitioning 
to renewable energy sources, improving energy efficiency, protecting and restoring forests, 
developing sustainable agriculture practices, and investing in carbon capture technologies. 
International agreements like the Paris Agreement aim to limit global warming to 1.5°C above 
pre-industrial levels.

Conclusion:
Climate change poses an existential threat that demands immediate and sustained action. While 
the challenges are immense, solutions exist and are increasingly economically viable. Success 
requires coordinated efforts from governments, businesses, and individuals worldwide.
"""

print(f"Document length: {len(long_document)} characters")
print(f"Approximate word count: {len(long_document.split())} words")


In [ ]:
# Basic summarization
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": f"""
Summarize the following document in 3-4 sentences, capturing the main points.

<document>
{long_document}
</document>
"""
    }]
)

print("📝 Basic Summary:")
show_response(response)


In [ ]:
# Structured summarization
response = client.messages.create(
    model="claude-sonnet-4-20250514",
    max_tokens=400,
    messages=[{
        "role": "user",
        "content": f"""
Analyze this document and provide a structured summary with:

1. **Main Topic**: One sentence description
2. **Key Statistics**: Any numbers or data mentioned
3. **Main Arguments**: 3 bullet points
4. **Conclusion**: The document's final message

<document>
{long_document}
</document>
"""
    }]
)

print("📊 Structured Summary:")
show_response(response)


In [ ]:
# Audience-specific summarization
audiences = [
    ("Executive", "Summarize for a busy CEO. Focus on business impact and action items. 2-3 sentences max."),
    ("Technical", "Summarize for a climate scientist. Include specific data and technical details."),
    ("General Public", "Summarize for a newspaper article. Use simple language, be engaging.")
]

print("👥 Audience-Specific Summaries:\n")

for audience, instruction in audiences:
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": f"""
{instruction}

<document>
{long_document}
</document>
"""
        }]
    )
    
    print(f"📌 {audience} Version:")
    print(response.content[0].text)
    print("\n" + "-"*50 + "\n")


### Handling Long Documents with Chunking

For documents exceeding the context window, we can use chunking strategies.


In [ ]:
def chunk_text(text, chunk_size=1000, overlap=100):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start = end - overlap
    return chunks

def summarize_long_document(document, client):
    """
    Summarize a long document using map-reduce approach:
    1. Split into chunks
    2. Summarize each chunk
    3. Combine summaries into final summary
    """
    # Split document into chunks
    chunks = chunk_text(document, chunk_size=1500, overlap=100)
    print(f"Document split into {len(chunks)} chunks")
    
    # Summarize each chunk
    chunk_summaries = []
    for i, chunk in enumerate(chunks):
        response = client.messages.create(
            model="claude-sonnet-4-20250514",
            max_tokens=150,
            messages=[{
                "role": "user",
                "content": f"Summarize this text section in 2-3 sentences:\n\n{chunk}"
            }]
        )
        summary = response.content[0].text
        chunk_summaries.append(summary)
        print(f"  Chunk {i+1}/{len(chunks)} summarized")
    
    # Combine summaries
    combined = "\n\n".join(chunk_summaries)
    
    # Final summary
    final_response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=250,
        messages=[{
            "role": "user",
            "content": f"""
Combine these section summaries into one coherent final summary:

{combined}

Provide a clear, well-structured summary in 4-5 sentences.
"""
        }]
    )
    
    return final_response.content[0].text

# Demo with our document
print("🔄 Map-Reduce Summarization:\n")
final_summary = summarize_long_document(long_document, client)
print("\n📄 Final Combined Summary:")
print(final_summary)


---
## Summary

In this segment, we covered:

1. **Basic Usage**: System prompts, assistant prefilling, and model selection
2. **Input Formatting**: XML tags, markdown, few-shot examples, chain-of-thought
3. **Multi-Step Prompts**: Conversation history, chatbots, sequential processing
4. **Document Summarization**: Basic, structured, audience-specific, and chunked approaches

### Best Practices:
- Use system prompts to set context and behavior
- Structure inputs with XML tags or markdown for clarity
- Provide examples for consistent output formats
- Break complex tasks into steps
- Choose summarization strategy based on document length and audience

### Next Steps
In Segment 3, we'll explore embeddings with Voyage AI for:
- Semantic search
- Question answering
- Recommendation systems
